# 🏺 Hieroglyph NLP Pipeline v4 — Full RAG Edition
### Gardiner Codes → Phonemes → RAG (bbaw 100k) → Seed-X → spaCy → Arabic
---
> **GPU: Tesla T4 (16GB)** | Seed-X-PPO-7B على GPU | BAAI/bge-m3 على CPU | Sentiment على CPU

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'⚠️  {result.stderr[-300:]}')

# numpy أولاً لمنع binary incompatibility
pip('numpy==1.26.4')

# Core
pip('protobuf==3.20.3')
pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
pip('flask', 'flask-cors', 'pyngrok')

# RAG dependencies
pip('faiss-cpu', 'sentence-transformers', 'datasets')

# spaCy
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')
print('⚠️  Restart the kernel now if first run (Run → Restart & Run All)')

## Cell 1 — Dataset Paths

In [ ]:
import os

GARDINER_PATH   = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv'
DICT_PATH       = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv'
INTENTION_PATH  = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv'

# الـ FAISS index هيتحفظ هنا عشان منعيدش نعمله كل مرة
FAISS_INDEX_PATH   = '/kaggle/working/bbaw_faiss.index'
FAISS_META_PATH    = '/kaggle/working/bbaw_meta.pkl'

for label, path in [('Gardiner', GARDINER_PATH), ('Dictionary', DICT_PATH), ('Intention', INTENTION_PATH)]:
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'  {status}  {label}: {path}')

print(f'\n  FAISS index cached: {"✅ YES" if os.path.exists(FAISS_INDEX_PATH) else "🔄 Will build on first run"}')

## Cell 2 — Load Gardiner Sign List

In [ ]:
import pandas as pd

df_g = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
for _, row in df_g.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic': str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning' : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode' : str(row['unicode']).strip()  if pd.notna(row['unicode'])  else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')

## Cell 3 — Load Intention Dataset

In [ ]:
df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }

print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')

## Cell 4 — Build RAG Index from bbaw_egyptian (100k)
> أول مرة بتاخد ~30-40 دقيقة. بعدين بتلود في ثواني من الـ cache.

In [ ]:
import pickle, faiss, numpy as np
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

print('🔄 Loading BAAI/bge-m3 embedding model on CPU...')
EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
print('✅ Embedding model loaded')

if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(FAISS_META_PATH):
    # ── لو الـ index موجود نلوده فوراً ──
    print('✅ Loading cached FAISS index...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f:
        bbaw_meta = pickle.load(f)
    print(f'   Index size: {faiss_index.ntotal} vectors')

else:
    # ── أول مرة: بنبني الـ index من الصفر ──
    print('🔄 Loading bbaw_egyptian dataset (100k)...')
    ds = load_dataset('phiwi/bbaw_egyptian', split='train')
    df_bbaw = ds.to_pandas()

    # بناخد الصفوف اللي فيها transcription و translation بس
    df_bbaw = df_bbaw[
        df_bbaw['transcription'].notna() &
        df_bbaw['translation'].notna()
    ].reset_index(drop=True)

    # بنعمل normalize بسيط للـ transcription
    def normalize_trans(t):
        import re
        t = str(t).lower()
        t = re.sub(r'[⸢⸣\[\]{}〈〉⸮\(\)]', '', t)  # شيل الـ textcritical symbols
        t = re.sub(r'\s+', ' ', t).strip()
        return t

    df_bbaw['trans_clean'] = df_bbaw['transcription'].apply(normalize_trans)

    print(f'   Dataset rows after cleaning: {len(df_bbaw)}')
    print('🔄 Generating embeddings (this takes ~30-40 min on CPU)...')

    # Batch encoding عشان يبقى أسرع
    BATCH_SIZE = 512
    all_texts = df_bbaw['trans_clean'].tolist()
    all_embeddings = EMBED_MODEL.encode(
        all_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    print('🔄 Building FAISS index...')
    dim = all_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)  # Inner Product (cosine بعد normalize)
    faiss_index.add(all_embeddings.astype(np.float32))

    # بنحفظ الـ index والـ metadata
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)

    bbaw_meta = {
        'transcriptions' : df_bbaw['trans_clean'].tolist(),
        'translations'   : df_bbaw['translation'].tolist(),
        'hieroglyphs'    : df_bbaw['hieroglyphs'].tolist() if 'hieroglyphs' in df_bbaw.columns else [''] * len(df_bbaw),
    }
    with open(FAISS_META_PATH, 'wb') as f:
        pickle.dump(bbaw_meta, f)

    print(f'✅ FAISS index built and cached: {faiss_index.ntotal} vectors')
    print(f'   Saved to: {FAISS_INDEX_PATH}')


def rag_search(query_text, top_k=5):
    """
    بيبحث في الـ 100k نص عن أقرب جمل للـ transliteration الجاية.
    بيرجع list من dict فيها transcription + translation (ألماني).
    """
    import re
    query_clean = re.sub(r'\s+', ' ', query_text.lower().strip())
    query_vec   = EMBED_MODEL.encode(
        [query_clean],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            'transcription': bbaw_meta['transcriptions'][idx],
            'translation'  : bbaw_meta['translations'][idx],
            'score'        : float(score),
        })
    return results


# ── Quick test ──
test_results = rag_search('nfr pr', top_k=3)
print('\n🔍 RAG Test: "nfr pr"')
for r in test_results:
    print(f'   [{r["score"]:.3f}] {r["transcription"][:40]} → {r["translation"][:60]}')

## Cell 5 — Load spaCy + Sentiment Models

In [ ]:
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

# ── spaCy ────────────────────────────────────────────────────
nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')

# ── Sentiment على CPU عشان نسيب الـ VRAM لـ Seed-X ──────────
print('🔄 Loading sentiment model on CPU...')
SENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer  = AutoTokenizer.from_pretrained(SENT_MODEL_NAME)
sent_model      = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
sent_model      = sent_model.to('cpu').eval()
print('✅ Sentiment model loaded (CPU)')

## Cell 6 — Load Seed-X on GPU

In [ ]:
from transformers import AutoTokenizer as ATok, AutoModelForSeq2SeqLM

SEED_X_MODEL = 'AlibabaResearch/Seed-X-PPO-7B'
print(f'🔄 Loading Seed-X on {DEVICE}...')

seedx_tokenizer = ATok.from_pretrained(SEED_X_MODEL)
seedx_model     = AutoModelForSeq2SeqLM.from_pretrained(
    SEED_X_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto'
)
seedx_model.eval()

if DEVICE == 'cuda':
    print(f'   VRAM used after load: {(torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1e9:.1f} GB')

print('✅ Seed-X loaded')


def translate_seedx(text, src_lang, tgt_lang, max_new_tokens=200):
    """
    يترجم باستخدام Seed-X.
    src_lang/tgt_lang: de, en, ar, fr ... إلخ
    """
    prompt = f'Translate the following {src_lang} text to {tgt_lang}:\n{text}'
    inputs = seedx_tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = seedx_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )
    return seedx_tokenizer.decode(out[0], skip_special_tokens=True).strip()


# ── Quick test ──
test_de  = 'Schönes Haus des Königs'
test_en  = translate_seedx(test_de, 'German', 'English')
test_ar  = translate_seedx(test_en, 'English', 'Arabic')
print(f'\n🔍 Seed-X Test:')
print(f'   DE: {test_de}')
print(f'   EN: {test_en}')
print(f'   AR: {test_ar}')

## Cell 7 — Core NLP Functions
### Gardiner → Phonemes → RAG → English → spaCy → Arabic

In [ ]:
import re

# ──────────────────────────────────────────────
# 7.1  Gardiner Codes → Phonemes (Sliding Window)
# ──────────────────────────────────────────────
def gardiner_to_phonemes(gardiner_codes_str):
    """
    بياخد string من Gardiner codes زي 'D21 N35 N5'
    وبيرجع:
      - phonemes: list من الأصوات ['r', 'n', 'ra']
      - assembled: الأصوات متجمعة 'r n ra'
      - unknown: الـ codes اللي مش عندنا ليها phonetic
    """
    codes = gardiner_codes_str.upper().split()
    phonemes = []
    unknown  = []

    for code in codes:
        key   = code.lower()
        entry = GARDINER_MAP.get(key, {})
        ph    = entry.get('phonetic', '').strip()
        if ph and ph.lower() not in ('nan', ''):
            phonemes.append(ph)
        else:
            unknown.append(code)

    assembled = ' '.join(phonemes)
    return {
        'phonemes' : phonemes,
        'assembled': assembled,
        'unknown'  : unknown,
        'codes'    : codes,
    }


# ──────────────────────────────────────────────
# 7.2  RAG → German Translation Context
# ──────────────────────────────────────────────
def get_rag_context(assembled_phonemes, top_k=5):
    """
    بيبحث في الـ bbaw 100k ويرجع أقرب جمل كـ context للـ Seed-X
    """
    results = rag_search(assembled_phonemes, top_k=top_k)
    context_lines = []
    for r in results:
        line = f'  Egyptian: {r["transcription"]}\n  German: {r["translation"]}'
        context_lines.append(line)
    return results, '\n\n'.join(context_lines)


# ──────────────────────────────────────────────
# 7.3  Seed-X: German context → English
# ──────────────────────────────────────────────
def rag_to_english(assembled_phonemes, rag_context_str):
    """
    بيديله الـ phonemes والـ RAG examples
    ويطلب منه يترجم إنجليزي
    """
    prompt = (
        f'You are an expert in Ancient Egyptian language translation.\n'
        f'Here are similar Egyptian texts with their German translations as reference:\n'
        f'{rag_context_str}\n\n'
        f'Now translate this Egyptian transliteration to English: "{assembled_phonemes}"\n'
        f'Provide only the English translation, nothing else.'
    )
    inputs = seedx_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = seedx_model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=4,
            early_stopping=True
        )
    return seedx_tokenizer.decode(out[0], skip_special_tokens=True).strip()


# ──────────────────────────────────────────────
# 7.4  spaCy: Fix Grammar & Complete Sentence
# ──────────────────────────────────────────────
def fix_with_spacy(raw_english):
    """
    بياخد الترجمة الإنجليزية الخام من الـ RAG
    ويستخدم spaCy لـ:
    1. يعرف الـ POS لكل كلمة
    2. يضيف missing articles/verbs لو محتاج
    3. يتأكد الجملة منطقية
    """
    doc = nlp_spacy(raw_english)

    # جمع المعلومات
    tokens_info = []
    for token in doc:
        tokens_info.append({
            'text' : token.text,
            'pos'  : token.pos_,
            'dep'  : token.dep_,
            'lemma': token.lemma_,
        })

    # فحص بسيط: هل في verb؟
    has_verb = any(t['pos'] in ('VERB', 'AUX') for t in tokens_info)
    has_subj = any(t['dep'] in ('nsubj', 'nsubjpass') for t in tokens_info)

    fixed = raw_english

    # لو مفيش verb ومفيش subject → جملة اسمية، نضيف 'is'
    if not has_verb and len(tokens_info) >= 2:
        # مثال: "beautiful house" → "The house is beautiful"
        nouns  = [t['text'] for t in tokens_info if t['pos'] == 'NOUN']
        adjs   = [t['text'] for t in tokens_info if t['pos'] == 'ADJ']
        if nouns and adjs:
            fixed = f'The {nouns[0]} is {" ".join(adjs)}'
        elif not raw_english.strip().endswith('.'):
            # على الأقل capitalize وضيف نقطة
            fixed = raw_english.strip().capitalize()

    # Capitalize أول حرف ونضيف نقطة لو مفيش
    fixed = fixed.strip().capitalize()
    if fixed and not fixed[-1] in '.!?':
        fixed += '.'

    return fixed, tokens_info


# ──────────────────────────────────────────────
# 7.5  Sentiment Analysis
# ──────────────────────────────────────────────
SENT_LABELS = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

def get_sentiment(text):
    inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = sent_model(**inputs).logits
    probs  = torch.softmax(logits, dim=-1)[0]
    pred   = int(probs.argmax())
    return {
        'label'      : SENT_LABELS[pred],
        'confidence' : float(probs[pred]),
        'scores'     : {
            'Negative': float(probs[0]),
            'Neutral' : float(probs[1]),
            'Positive': float(probs[2]),
        }
    }


# ──────────────────────────────────────────────
# 7.6  Intention Detection
# ──────────────────────────────────────────────
def get_intention(english_text):
    words = set(english_text.lower().split())
    best_intent  = 'unknown'
    best_count   = 0
    best_ar      = ''

    for intent, data in INTENTION_MAP.items():
        overlap = len(words & data['keywords'])
        if overlap > best_count:
            best_count  = overlap
            best_intent = intent
            best_ar     = data.get('arabic', '')

    return {'intention_en': best_intent, 'intention_ar': best_ar, 'keyword_matches': best_count}


print('✅ All NLP functions defined')

## Cell 8 — Master Pipeline Function

In [ ]:
import time

def full_pipeline(gardiner_codes_str, verbose=True):
    """
    الـ Master function:
    Input:  'D21 N35 N5'  (Gardiner codes)
    Output: dict فيه كل النتايج
    """
    t0 = time.time()
    result = {'input': gardiner_codes_str, 'steps': {}}

    # ── Step 1: Gardiner → Phonemes ──────────────────────────
    phon = gardiner_to_phonemes(gardiner_codes_str)
    result['phonemes']   = phon['phonemes']
    result['assembled']  = phon['assembled']
    result['unknown']    = phon['unknown']
    result['steps']['phonemes'] = f"{' '.join(phon['phonemes'])} ({len(phon['unknown'])} unknown)"
    if verbose:
        print(f'\n📜 Step 1 — Phonemes: {phon["assembled"]}')
        if phon['unknown']:
            print(f'   ⚠️  Unknown codes: {phon["unknown"]}')

    # ── Step 2: RAG Search ───────────────────────────────────
    if verbose: print('🔍 Step 2 — RAG Search in bbaw 100k...')
    rag_results, rag_ctx = get_rag_context(phon['assembled'], top_k=5)
    result['rag_results'] = rag_results
    if verbose:
        for i, r in enumerate(rag_results[:3]):
            print(f'   [{i+1}] score={r["score"]:.3f} | {r["transcription"][:35]} → {r["translation"][:50]}')

    # ── Step 3: RAG → English (Seed-X) ──────────────────────
    if verbose: print('🌍 Step 3 — Seed-X: context → English...')
    raw_english = rag_to_english(phon['assembled'], rag_ctx)
    result['steps']['raw_english'] = raw_english
    if verbose: print(f'   Raw EN: {raw_english}')

    # ── Step 4: spaCy Grammar Fix ────────────────────────────
    if verbose: print('🔧 Step 4 — spaCy grammar fix...')
    fixed_english, tokens_info = fix_with_spacy(raw_english)
    result['english']      = fixed_english
    result['tokens']       = tokens_info
    result['steps']['fixed_english'] = fixed_english
    if verbose: print(f'   Fixed EN: {fixed_english}')

    # ── Step 5: English → Arabic (Seed-X) ───────────────────
    if verbose: print('🌙 Step 5 — Seed-X: English → Arabic...')
    arabic_text = translate_seedx(fixed_english, 'English', 'Arabic')
    result['arabic'] = arabic_text
    if verbose: print(f'   AR: {arabic_text}')

    # ── Step 6: Sentiment ────────────────────────────────────
    sentiment = get_sentiment(fixed_english)
    result['sentiment'] = sentiment

    # ── Step 7: Intention ────────────────────────────────────
    intention = get_intention(fixed_english)
    result['intention'] = intention

    result['processing_time'] = round(time.time() - t0, 2)

    if verbose:
        print(f'\n{'='*60}')
        print(f'✅ DONE in {result["processing_time"]}s')
        print(f'   🏺 Codes    : {gardiner_codes_str}')
        print(f'   🔤 Phonemes : {phon["assembled"]}')
        print(f'   🇬🇧 English : {fixed_english}')
        print(f'   🌙 Arabic   : {arabic_text}')
        print(f'   💬 Sentiment: {sentiment["label"]} ({sentiment["confidence"]:.2%})')
        print(f'   🎯 Intention: {intention["intention_en"]}')
        print(f'{'='*60}')

    return result


# ── Quick Test ──
print('🧪 Testing pipeline with: D21 N35 N5')
test_result = full_pipeline('D21 N35 N5')

## Cell 9 — Flask API + Ngrok

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

app = Flask(__name__)
CORS(app)


@app.route('/api/translate', methods=['POST'])
def api_translate():
    data = request.get_json()
    gardiner = data.get('gardiner', '').strip()
    if not gardiner:
        return jsonify({'error': 'gardiner field is required'}), 400
    try:
        result = full_pipeline(gardiner, verbose=False)
        return jsonify({
            'success'    : True,
            'input'      : result['input'],
            'phonemes'   : result['assembled'],
            'english'    : result['english'],
            'arabic'     : result['arabic'],
            'sentiment'  : result['sentiment'],
            'intention'  : result['intention'],
            'rag_results': result['rag_results'][:3],
            'time'       : result['processing_time'],
        })
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500


@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': 'v4-RAG'})


@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify([
        {'name': 'Name of Ra', 'codes': 'D21 N35 N5'},
        {'name': 'Beautiful House', 'codes': 'F35 O1'},
        {'name': 'King of Egypt', 'codes': 'L2 N17'},
        {'name': 'Offering to Osiris', 'codes': 'R4 Q1'},
    ])


# ── Start ────────────────────────────────────
NGROK_TOKEN = ''  # ← حط الـ token بتاعك هنا

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print(f'🌍 Public URL: {public_url}')
    print(f'   POST {public_url}/api/translate')
else:
    print('⚠️  No ngrok token — running on localhost only')

def run_app():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()
print('✅ Flask API running on port 5000')

## Notes & Pipeline Summary

```
INPUT: Gardiner Codes (e.g. 'D21 N35 N5')
       ↓
Step 1: Gardiner → Phonemes
        D21→r, N35→n, N5→ra  =>  'r n ra'
       ↓
Step 2: FAISS RAG Search (bbaw 100k, BAAI/bge-m3 embeddings)
        Top 5 أقرب جمل مصرية مع ترجمتها الألمانية
       ↓
Step 3: Seed-X (GPU) — German context + phonemes → English
        'r n ra' + context → 'The name of Ra'
       ↓
Step 4: spaCy — Grammar fix + add missing words
        'name Ra' → 'The name of Ra.'
       ↓
Step 5: Seed-X (GPU) — English → Arabic
        'The name of Ra.' → 'اسم رع'
       ↓
Step 6: Sentiment (CPU) — Positive/Neutral/Negative
Step 7: Intention detection — prayer/offering/royal/etc.
       ↓
OUTPUT: EN + AR + Sentiment + Intention + RAG context
```

### VRAM Budget (T4 16GB)
| Component | Location | Usage |
|-----------|----------|-------|
| Seed-X 7B | GPU | ~14 GB |
| BAAI/bge-m3 | CPU | ~1.5 GB RAM |
| Sentiment | CPU | ~0.5 GB RAM |
| FAISS index | CPU RAM | ~2-3 GB RAM |

### First Run Time
- Embedding 100k texts: ~30-40 min (مرة واحدة بس)
- بعدين من الـ cache: < 5 ثواني